In [7]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import pymysql
import csv
import yaml

EPA

In [ ]:
email = "josea@clarkson.edu"
api_key = "russetwolf34" 
param = "88101"
locations = {
    "Albany, NY": ("36", "001"),
    "Syracuse, NY": ("36", "067"),
}
date_ranges = [
    ("20230101","20231231"),
    ("20240101", "20241231"),
    ("20250101", "20251201")
]

all_data = []
for city, (state, county) in locations.items():
    for startdate, enddate in date_ranges:
        params = {
            "email": email,
            "key": api_key,
            "param": param,
            "bdate": startdate,
            "edate": enddate,
            "state": state,
            "county": county
        }

        response = requests.get("https://aqs.epa.gov/data/api/dailyData/byCounty", 
                                params=params)
        data = response.json()
        if "Data" in data:
            df = pd.DataFrame(data["Data"])
            df["city"] = city
            all_data.append(df)
        else:
            print("No data returned.")

In [27]:
pm25_df = pd.concat(all_data, ignore_index=True)
pm25_df["date_local"] = pd.to_datetime(pm25_df["date_local"])

In [28]:
pm25_df = pm25_df[['date_local', 'arithmetic_mean', 'aqi', 'state', 'county']]
pm25_df.rename(columns={'arithmetic_mean':'PM25'},inplace=True)
pm25_df = pm25_df.groupby(['date_local', 'county'], as_index=False).first()
pm25_df["city"] = pm25_df["county"].replace({"Onondaga": "Syracuse"})
pm25_df = pm25_df.drop('county',axis = 1)
pm25_df.to_csv("pm25_by_county_23_25.csv", index=False)
pm25_df.head(5)

,date_local,PM25,aqi,state,city
0,2023-01-01,7.275000,40.0,New York,Albany
1,2023-01-01,9.200000,51.0,New York,Syracuse
2,2023-01-02,13.941667,60.0,New York,Albany
3,2023-01-02,11.600000,56.0,New York,Syracuse
4,2023-01-03,14.000000,60.0,New York,Albany


NOAA

In [5]:
stations_dict = {
    "Albany": "GHCND:USW00014735",
    "Syracuse": "GHCND:USW00014771",
}
datasetid = "GHCND"
headers = {"token": "qwsiEPAxqjCiDPsnKiAVWJBPCecAhwNU"}
datatypeid = ["TMAX", "TMIN", "PRCP","SNWD","AWND","RHMN"] # wind direction
date_ranges = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-01")
]
all_data = []

for city, stationid in stations_dict.items():
    for startdate, enddate in date_ranges:
        limit = 1000
        offset = 1
        
        while True:
            params = {
                "datasetid": datasetid,
                "stationid": stationid,
                "startdate": startdate,
                "enddate": enddate,
                "datatypeid": datatypeid,
                "limit": limit,
                "offset": offset,
                "units": "standard"
            }

            response = requests.get(
                "https://www.ncdc.noaa.gov/cdo-web/api/v2/data",
                headers=headers,
                params=params
            )

            data = response.json()
            for rec in data["results"]:
                rec["city"] = city
            all_data.extend(data["results"])
            
            if len(data["results"]) < limit:
                break
            offset += limit


In [14]:
df = pd.DataFrame(all_data)
df["date"] = pd.to_datetime(df["date"])
df_pivot = df.pivot_table(index=["date", "city"], columns="datatype", values="value")
df_pivot.reset_index(inplace=True)
#df_pivot = df_pivot.drop('datatype',axis=1)
df_pivot.to_csv('weather_data_23-25.csv',index= False)
df_pivot.head()

datatype,date,city,AWND,PRCP,RHMN,SNWD,TMAX,TMIN
0,2023-01-01,Albany,10.3,0.0,62.0,0.0,51.0,34.0
1,2023-01-01,Syracuse,8.3,0.0,65.0,0.0,50.0,38.0
2,2023-01-02,Albany,2.0,0.0,59.0,0.0,51.0,29.0
3,2023-01-02,Syracuse,4.9,0.0,68.0,0.0,46.0,39.0
4,2023-01-03,Albany,0.4,0.3,89.0,0.0,37.0,27.0


Wind speed

In [4]:
cities = {
    "Syracuse": {"lat": 43.0481, "lon": -76.1474},
    "Albany": {"lat": 42.6526, "lon": -73.7562}
}

start_date = "2023-01-01"
end_date = "2025-12-01"

all_data = []

for city, coords in cities.items():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "daily": "wind_direction_10m_dominant",
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    data = response.json().get("daily", {})
    
    dates = data.get("time", [])
    wind_dirs = data.get("wind_direction_10m_dominant", [])
    
    for d, wd in zip(dates, wind_dirs):
        all_data.append({"date": d, "city": city, "wind_direction": wd})

wind_df = pd.DataFrame(all_data)
wind_df = wind_df.sort_values(["date", "city"]).reset_index(drop=True)


In [6]:
wind_df.to_csv('Wind_direction.csv',index=False)
wind_df.head()

,date,city,wind_direction
0,2023-01-01,Albany,306
1,2023-01-01,Syracuse,264
2,2023-01-02,Albany,165
3,2023-01-02,Syracuse,248
4,2023-01-03,Albany,278


Data Processing

In [7]:
weather_df = pd.read_csv('weather_data_23-25.csv',index_col=False)
pm25_df = pd.read_csv('pm25_by_county_23_25.csv',index_col=False)
wind_df = pd.read_csv('Wind_direction.csv',index_col=False)

pm25_df.rename(columns={'date_local':'date','county':'city'},inplace=True)

weather_df["date"] = pd.to_datetime(weather_df["date"])
pm25_df["date"] = pd.to_datetime(pm25_df["date"])
wind_df["date"] = pd.to_datetime(wind_df["date"])

df_temp = pd.merge(weather_df, pm25_df, on=['date','city'], how='inner')
df = pd.merge(df_temp, wind_df, on=['date','city'], how='inner')
df.head()

,date,city,AWND,PRCP,RHMN,SNWD,TMAX,TMIN,PM25,aqi,state,wind_direction
0,2023-01-01,Albany,10.3,0.0,62.0,0.0,51.0,34.0,7.275000,40.0,New York,306
1,2023-01-01,Syracuse,8.3,0.0,65.0,0.0,50.0,38.0,9.200000,51.0,New York,264
2,2023-01-02,Albany,2.0,0.0,59.0,0.0,51.0,29.0,13.941667,60.0,New York,165
3,2023-01-02,Syracuse,4.9,0.0,68.0,0.0,46.0,39.0,11.600000,56.0,New York,248
4,2023-01-03,Albany,0.4,0.3,89.0,0.0,37.0,27.0,14.000000,60.0,New York,278


In [8]:
df.to_csv('All_data.csv',index=False)

pymysql

In [ ]:
with open('config.yml') as f:
    db = yaml.safe_load(f)['db']

In [ ]:
conn = pymysql.connect(host=db['host'],port=3306,user=db['user'],
                       passwd = db['pw'],db = db['db'],autocommit=True)
cur = conn.cursor(pymysql.cursors.DictCursor)

cur.execute('DROP TABLE IF EXISTS `weather_air_quality`;')
table = """
CREATE TABLE `weather_air_quality` (
    `tid` INT AUTO_INCREMENT PRIMARY KEY,
    `date` DATE NOT NULL,
    `city` VARCHAR(25) NOT NULL,
    `state` VARCHAR(50),
    `wind_speed` FLOAT,
    `precipitation` FLOAT,
    `relative_humidity` FLOAT,
    `snow` FLOAT,
    `temperature_max` FLOAT,
    `temperature_min` FLOAT,
    `pm25` FLOAT,
    `aqi` FLOAT,
    `wind_direction` INT,
    UNIQUE KEY uniq_weather (`date`,`city`)
);
"""
cur.execute(table)

0

In [14]:
def clean_float(val):
    try:
        if val in ('','NA','N/A','null',None):
            return None
        return float(val)
    except:
        return None

In [15]:
data = []

with open('All_data.csv', 'r') as f:
    for row in csv.DictReader(f, skipinitialspace=True):
        data.append((
            row['date'],row['city'],row['state'],
            clean_float(row['AWND']),clean_float(row['PRCP']),
            clean_float(row['RHMN']),clean_float(row['SNWD']),
            clean_float(row['TMAX']),clean_float(row['TMIN']),clean_float(row['PM25']),
            clean_float(row['aqi']),clean_float(row['wind_direction'])
        ))

insert_weather = """
INSERT INTO `weather_air_quality`(
    `date`,`city`,`state`,`wind_speed`,`precipitation`,`relative_humidity`,`snow`,
    `temperature_max`,`temperature_min`,`pm25`,`aqi`,`wind_direction`
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

cur.executemany(insert_weather, data)

2000